# ManualChaidTree — manual demo

Hand-build a CHAID scorecard tree on the Kaggle **Default of Credit Card Clients** dataset.

1. Download the dataset from https://www.kaggle.com/datasets/uciml/default-of-credit-card-clients-dataset
2. Place `UCI_Credit_Card.csv` in the project root (one folder above this notebook).
3. Run the cells below.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
from sklearn.model_selection import train_test_split
from manual_chaid import ManualChaidTree

df = pd.read_csv('../UCI_Credit_Card.csv').drop(columns=['ID'])
y = df.pop('default.payment.next.month')
X = df

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)
X_train = X_train.reset_index(drop=True); y_train = y_train.reset_index(drop=True)
X_test  = X_test.reset_index(drop=True);  y_test  = y_test.reset_index(drop=True)
print(X_train.shape, 'target rate =', round(y_train.mean(), 4))
X_train.head()

## 1. Launch the interactive builder

`train()` opens the tkinter window (canvas + side menu). `PAY_0..PAY_6` are repayment-status
columns where `0` and `-1` carry special meaning, so we pre-separate them.

Build the tree in the window: pick a leaf, choose a high-χ² variable, tweak cut-offs, **Split**;
then **Combine** / **Prune** / **Undo** as needed; finally **Save** and **Export** from the Actions panel.
Close the window to return here.

In [ ]:
tree = ManualChaidTree(X_train, y_train)
tree.train(
    pre_separate_zero=['PAY_0', 'PAY_2', 'PAY_3'],
    pre_separate_values={'PAY_0': -1, 'PAY_2': -1},
    min_samples=500,
    max_children=4,
)

## 2. (Optional) Build headlessly to test the core without the UI

Everything the UI does is available on the object. Pass `launch_ui=False` and drive it from code.

In [ ]:
def show_tree(t):
    for nid in sorted(t.nodes):
        n = t.nodes[nid]
        pad = '  ' * n.depth
        rule = n.condition.describe() if n.condition else 'ROOT'
        extra = ''
        if not n.is_leaf:
            extra = f"  | split={n.split_variable} chi2={n.split_chi2:.1f} p={n.split_pvalue:.2g}"
        print(f"{pad}#{nid} [{rule}] n={n.n_samples} bad={n.target_rate:.1%}{extra}")

t = ManualChaidTree(X_train, y_train)
t.train(pre_separate_zero=['PAY_0'], pre_separate_values={'PAY_0': -1},
        min_samples=500, max_children=4, launch_ui=False)

# Rank candidate variables for the root by chi-square:
cands = t.candidate_splits(0)
for c in cands[:6]:
    print(f"{c['variable']:<10} chi2={c['chi2']:.1f}  p={c['pvalue']:.2g}  splittable={c['splittable']}")

In [ ]:
# Split the root on the top candidate, then split a child, then inspect.
top = cands[0]
t.split(0, top['variable'], top['conditions'])

leaf = max(t.leaves(), key=lambda n: n.n_samples)
child_cands = t.candidate_splits(leaf.node_id)
best = next((c for c in child_cands if c['splittable']), None)
if best:
    t.split(leaf.node_id, best['variable'], best['conditions'])

show_tree(t)
print('\nhistory:', t.history)

In [ ]:
# Combine / prune / undo round-trip
root_children = t.nodes[0].children
if len(root_children) >= 2:
    leaf_kids = [c for c in root_children if t.nodes[c].is_leaf][:2]
    if len(leaf_kids) == 2:
        t.combine(0, leaf_kids)
        print('after combine:', t.history[-1])
        t.undo()
        print('after undo, history:', t.history)

## 3. Gini, save / load, and scoring new data

In [ ]:
print('Train Gini:', round(t.gini(), 4))
print('Test  Gini:', round(t.gini(X_test, y_test), 4))

t.save('tree.pkl')
print('saved size (bytes):', os.path.getsize('tree.pkl'))

# Reload WITH data -> editable; WITHOUT data -> view / score only.
t_edit = ManualChaidTree.load('tree.pkl', X=X_train, y=y_train)
print('reloaded (with data) train Gini:', round(t_edit.gini(), 4))

t_view = ManualChaidTree.load('tree.pkl')
print('scored new rows (no data needed):', t_view.predict_proba(X_test.head()))

## 4. Export the leaf rule set to Excel

In [ ]:
rules = t.export_rules_excel('leaf_rules.xlsx')
rules